This is the code for the final project in ECSE 343: Numerical Methods for Engs. Our team, group 20, is made up of
- Serge Al Laham
- Emily Wang
- Jeremias Zimmerman
- Steven Thao

# Part 1: Data generation

# Part 2: Newton-Raphson method

# Part 3: ML solution

In [25]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

"""
Class for simple feedforward neural network.
"""
class NeuralNet(nn.Module):
    """
    For now, the activation function is ReLU. Can adapt to other functions in the future.

    Input:
        units: list of number of units in each layer, including input and output layers.
    Output:
        NeuralNet object.
    """
    def __init__(self, units: list[int], lr: float = 0.001) -> None:
        super().__init__()
        self.hidden = nn.ModuleList([nn.Linear(units[i], units[i + 1]) for i in range(len(units) - 1)])
        self.activation = nn.ReLU()
        self.optimizer = optim.Adam(self.parameters(), lr=lr)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.hidden[:-1]:
            x = self.activation(layer(x))
        return self.hidden[-1](x)

In [9]:
def split_dataset(dataset: tuple[torch.Tensor, torch.Tensor], train_ratio: float = 0.7, val_ratio: float = 0.15) -> tuple[tuple[torch.Tensor, torch.Tensor], tuple[torch.Tensor, torch.Tensor], tuple[torch.Tensor, torch.Tensor]]:
    """
    Split dataset into training, validation, and test sets.

    Input:
        dataset: single tuple of (X, y) where X is all the input data and y is all the labels.
        train_ratio: decimal ratio of training data.
        val_ratio: decimal ratio of validation data.
    Output:
        tuple of (X_train, y_train), (X_val, y_val), (X_test, y_test) where X and y are torch tensors.
    """
    X, y = dataset
    n_samples = X.shape[0]
    # shuffle indices
    indices = np.random.permutation(n_samples)
    # calculate split indices
    train_end = int(train_ratio * n_samples)
    val_end = int((train_ratio + val_ratio) * n_samples)

    # do the split
    X_train, y_train = X[indices[:train_end]], y[indices[:train_end]]
    X_val, y_val = X[indices[train_end:val_end]], y[indices[train_end:val_end]]
    X_test, y_test = X[indices[val_end:]], y[indices[val_end:]]

    return (torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)), (torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)), (torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32))

In [30]:
# TESTING: Assumed dataset:
# dataset = {(x^(i), y^(i))} for i in [1, N]
# In reality, dataset = (X, y), with X being an N x 26 matrix, and y being an N x 2 matrix.
# y^{(i)} = (R^{(i)}, C^{(i)})
# x^{(i)} = 26-length real-valued vector of features for the ith simulation.

from tqdm import tqdm

# 1. Split dataset into training, validation, and test sets.
# example dataset with 100 samples, 26 features, and 2 labels
dataset = (np.random.rand(100, 26), np.random.rand(100, 2))
D_train, D_val, D_test = split_dataset(dataset, train_ratio=0.7, val_ratio=0.15)
print("Training set:", D_train[0].shape, D_train[1].shape)
print("Validation set:", D_val[0].shape, D_val[1].shape)
print("Test set:", D_test[0].shape, D_test[1].shape)

# 2. Initialize neural network.
units = [26, 64, 32, 2] # 26 inputs, 2 hidden layers, 2 outputs
model = NeuralNet(units, lr=0.001)
print(model)

# 3. Train neural network on training set, using validation set for early stopping.
criterion = nn.MSELoss()
best_val_loss = float('inf')
patience = 10
counter = 0
epochs = 100

for epoch in tqdm(range(epochs), desc="Training", unit="epoch"):  # max epochs
    model.train()
    model.optimizer.zero_grad()
    outputs = model(D_train[0])
    loss = criterion(outputs, D_train[1])
    loss.backward()
    model.optimizer.step()

    # validate
    model.eval()
    with torch.no_grad():
        val_outputs = model(D_val[0])
        val_loss = criterion(val_outputs, D_val[1])

    tqdm.write(f"Epoch {epoch}, Training Loss: {loss.item()}, Validation Loss: {val_loss.item()}")

    # early stopping
    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            tqdm.write("Early stopping triggered.")
            break

Training set: torch.Size([70, 26]) torch.Size([70, 2])
Validation set: torch.Size([15, 26]) torch.Size([15, 2])
Test set: torch.Size([15, 26]) torch.Size([15, 2])
NeuralNet(
  (hidden): ModuleList(
    (0): Linear(in_features=26, out_features=64, bias=True)
    (1): Linear(in_features=64, out_features=32, bias=True)
    (2): Linear(in_features=32, out_features=2, bias=True)
  )
  (activation): ReLU()
)


Training:  80%|████████  | 80/100 [00:00<00:00, 457.74epoch/s]

Epoch 0, Training Loss: 0.47542300820350647, Validation Loss: 0.40921688079833984
Epoch 1, Training Loss: 0.4552431106567383, Validation Loss: 0.3909936249256134
Epoch 2, Training Loss: 0.43654176592826843, Validation Loss: 0.37383729219436646
Epoch 3, Training Loss: 0.41870152950286865, Validation Loss: 0.3578292429447174
Epoch 4, Training Loss: 0.4016343951225281, Validation Loss: 0.34208542108535767
Epoch 5, Training Loss: 0.38485461473464966, Validation Loss: 0.326417475938797
Epoch 6, Training Loss: 0.36834678053855896, Validation Loss: 0.31083962321281433
Epoch 7, Training Loss: 0.35198304057121277, Validation Loss: 0.2951779067516327
Epoch 8, Training Loss: 0.33572468161582947, Validation Loss: 0.279469758272171
Epoch 9, Training Loss: 0.31939762830734253, Validation Loss: 0.26385536789894104
Epoch 10, Training Loss: 0.30309757590293884, Validation Loss: 0.24874736368656158
Epoch 11, Training Loss: 0.28693968057632446, Validation Loss: 0.23412786424160004
Epoch 12, Training Loss